# BensLens DSPL Workflow


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'Tian_infra.py').exists():
    for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
        if (candidate / 'Tian_infra.py').exists():
            NOTEBOOK_DIR = candidate
            break
if not (NOTEBOOK_DIR / 'Tian_infra.py').exists():
    raise FileNotFoundError('Cannot locate Tian_infra.py from the current working directory.')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from Tian_infra import import_function_DSPL
import_function_DSPL(globals())

NOTEBOOK_DIR


In [ ]:
PREPROCESSED_DIR = NOTEBOOK_DIR / 'preprocessed' / 'prep_20260420_benslens_v1'
PREPROCESS_META_PATH = PREPROCESSED_DIR / 'preprocessing_metadata.json'
PREPROCESS_META = json.loads(PREPROCESS_META_PATH.read_text()) if PREPROCESS_META_PATH.exists() else None

CONJUGATE_JSON_PATH = None
CONJUGATE_REFERENCE_BAND = None
for band_key in ['f814w', 'f475w']:
    candidate = PREPROCESSED_DIR / f'{band_key}_lens_cutout_220x220_conjugate_points.json'
    if candidate.exists():
        CONJUGATE_JSON_PATH = candidate
        CONJUGATE_REFERENCE_BAND = band_key
        break
if CONJUGATE_JSON_PATH is None:
    raise FileNotFoundError('No conjugate-point JSON found in preprocessed directory.')
CONJUGATE_PAYLOAD = json.loads(CONJUGATE_JSON_PATH.read_text())

TRACE_POINT_JSON_PATH = None
TRACE_POINT_PAYLOAD = None
TRACE_POINT_REFERENCE_BAND = None

TRACE_CONJUGATE_GROUPS = {
    'source1': [3, 4],
    'source2': [1, 2, 8],
}
USE_TRACE_POINT_CONJUGATE_PRIOR = False
CONJUGATE_PRIOR_RATE = 1000.0

PIXEL_GRID_SHAPE = 50
PARAMETRIC_STEPS = 10000
PIXELATED_STEPS = 10000
STAGE2_INIT_STEPS = 200
RNG_SEED = 7
N_GAUSS_LENS = 15
N_GAUSS_SOURCE = 4
PRIORS = {
    'theta_E_main': {'low': 0.01, 'high': 2.03},
    'gamma_main': {'low': 1.6, 'high': 2.4},
    'q_main': {'low': 0.5, 'high': 1},
    'gamma_shear_main': {'low': -0.15, 'high': 0.15},
    'center_main': {'low': -0.4, 'high': 0.4},
    'phi_main': {'low': -0.5 * np.pi, 'high': 0.5 * np.pi},
    'theta_E_source1': {'low': 0.00, 'high': 0.3},
    'eta': {'low': 1.0, 'high': 1.5},
}
SOURCE_GRID_SCALE = 1.0
USE_SOURCE_MASKS_PARAMETRIC = False
USE_SOURCE_MASKS_PIXELATED = True
ACTIVE_BANDS = ['f475w', 'f814w']
REFERENCE_BAND = 'f814w'
if REFERENCE_BAND not in ACTIVE_BANDS:
    raise ValueError(f'REFERENCE_BAND {REFERENCE_BAND} must be included in ACTIVE_BANDS {ACTIVE_BANDS}')
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'benslens_notebook_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PREPROCESS_META is None:
    raise FileNotFoundError(f'Missing preprocessing metadata: {PREPROCESS_META_PATH}')

CUTOUT_SIZE = tuple(PREPROCESS_META['science_cutout_size'])
INNER_MASK_PATH = PREPROCESSED_DIR / 'f814w_lens_cutout_220x220_mask.fits'
OUTER_MASK_PATH = PREPROCESSED_DIR / 'f475w_lens_cutout_220x220_mask.fits'

BAND_FILES = {
    'f475w': {
        'science_path': PREPROCESSED_DIR / 'f475w_lens_cutout_220x220.fits',
        'psf_path': PREPROCESSED_DIR / 'f475w_psf_model_svi.fits',
        'background_box_path': PREPROCESSED_DIR / 'f475w_background_box_50x50.fits',
    },
    'f814w': {
        'science_path': PREPROCESSED_DIR / 'f814w_lens_cutout_220x220.fits',
        'psf_path': PREPROCESSED_DIR / 'f814w_psf_model_svi.fits',
        'background_box_path': PREPROCESSED_DIR / 'f814w_background_box_50x50.fits',
    },
}

with fits.open(BAND_FILES[ACTIVE_BANDS[0]]['science_path'], memmap=True) as hdul:
    REFERENCE_SCIENCE_WCS = WCS(hdul[0].header)

with fits.open(BAND_FILES[ACTIVE_BANDS[0]]['psf_path'], memmap=True) as hdul:
    PSF_CUTOUT_SIZE = tuple(np.asarray(hdul[0].data).shape)

for band_key in ACTIVE_BANDS:
    band_files = BAND_FILES[band_key]
    for file_key, file_path in band_files.items():
        print(f'{band_key} {file_key}: {file_path}')
print('inner mask:', INNER_MASK_PATH)
print('outer mask:', OUTER_MASK_PATH)
print('conjugate json:', CONJUGATE_JSON_PATH)
print('conjugate reference band:', CONJUGATE_REFERENCE_BAND)
print('trace point json:', 'not used')




In [ ]:
@dataclass
class BandData:
    key: str
    filter_name: str
    data: jnp.ndarray
    obs_mask: jnp.ndarray
    num_obs: int
    source1_mask: np.ndarray
    source2_mask: np.ndarray
    rms: float
    pixel_scale: float
    exposure_time: float
    science_cutout_path: Path
    source1_mask_cutout_path: Path
    source2_mask_cutout_path: Path
    psf_cutout_path: Path
    background_box_path: Path
    science_extent: tuple[float, float, float, float]
    psf_extent: tuple[float, float, float, float]
    science_wcs: WCS
    lens_image_parametric: MPLensImage
    lens_image_pixelated: MPLensImage
    conjugate_points_model: np.ndarray | None


def pixel_scale_arcsec(header):
    if 'PIXSCALE' in header:
        return float(header['PIXSCALE'])
    if 'D001SCAL' in header:
        return float(header['D001SCAL'])
    cd11 = header.get('CD1_1')
    cd21 = header.get('CD2_1')
    if cd11 is not None and cd21 is not None:
        return float(np.hypot(cd11, cd21) * 3600.0)
    raise ValueError('Cannot determine pixel scale from FITS header.')


def load_binary_mask(path):
    with fits.open(path, memmap=True) as hdul:
        mask = np.asarray(hdul[0].data, dtype=np.uint8)
    return mask > 0


def load_points_from_payload(payload, key):
    if payload is None:
        return None
    points = payload.get(key, [])
    if len(points) == 0:
        return None
    return np.array([[point['x'], point['y']] for point in points], dtype=np.float64)


def pixel_fits_to_model_xy_array(points_xy, shape, pixel_scale):
    points_xy = np.asarray(points_xy, dtype=np.float64)
    center_xy = np.array([(shape[1] - 1) / 2.0, (shape[0] - 1) / 2.0], dtype=np.float64)
    return (points_xy - center_xy) * pixel_scale


def map_fits_points_between_wcs(points_xy, source_wcs, target_wcs):
    points_xy = np.asarray(points_xy, dtype=np.float64)
    world = source_wcs.pixel_to_world_values(points_xy[:, 0], points_xy[:, 1])
    x_target, y_target = target_wcs.world_to_pixel_values(*world)
    return np.column_stack([x_target, y_target])


def extent_axes(extent, shape):
    ny, nx = shape
    x_axis = np.linspace(extent[0], extent[1], nx)
    y_axis = np.linspace(extent[2], extent[3], ny)
    return x_axis, y_axis


SOURCE1_MASK = load_binary_mask(INNER_MASK_PATH)
SOURCE2_MASK = load_binary_mask(OUTER_MASK_PATH)
if SOURCE1_MASK.shape != tuple(CUTOUT_SIZE):
    raise ValueError(f'Inner mask shape {SOURCE1_MASK.shape} does not match cutout {CUTOUT_SIZE}')
if SOURCE2_MASK.shape != tuple(CUTOUT_SIZE):
    raise ValueError(f'Outer mask shape {SOURCE2_MASK.shape} does not match cutout {CUTOUT_SIZE}')


def build_band_data(key):
    band_meta = PREPROCESS_META['cutouts'][key]
    band_files = BAND_FILES[key]
    science_cutout_path = band_files['science_path']
    psf_cutout_path = band_files['psf_path']
    background_box_path = band_files['background_box_path']

    with fits.open(science_cutout_path, memmap=True) as hdul:
        science_header = hdul[0].header
        science_data = np.asarray(hdul[0].data, dtype=np.float64)
    science_wcs = WCS(science_header)
    with fits.open(psf_cutout_path, memmap=True) as hdul:
        psf_header = hdul[0].header
        psf_kernel = np.asarray(hdul[0].data, dtype=np.float64)

    filter_name = science_header.get('FILTER', key.upper())
    exposure_time = float(band_meta['exptime_sec'])
    pixel_scale = pixel_scale_arcsec(science_header)
    rms = float(band_meta['background_sigma_scalar'])

    obs_mask = np.isfinite(science_data)
    science_data = np.nan_to_num(science_data, nan=0.0).astype(np.float64)
    source1_mask = np.asarray(SOURCE1_MASK, dtype=bool)
    source2_mask = np.asarray(SOURCE2_MASK, dtype=bool)

    psf_kernel = np.nan_to_num(psf_kernel, nan=0.0).astype(np.float64)
    psf_kernel = np.clip(psf_kernel, a_min=0.0, a_max=None)
    psf_kernel /= psf_kernel.sum()

    pixel_grid, _, _, _, _, science_extent, _, _ = Geometry.get_pixel_grid(science_data, pixel_scale)
    psf_extent = Geometry.get_pixel_grid(psf_kernel, pixel_scale)[5]
    psf = PSF(psf_type='PIXEL', kernel_point_source=psf_kernel)
    noise = Noise(
        pixel_grid.num_pixel_axes[1],
        pixel_grid.num_pixel_axes[0],
        exposure_time=exposure_time,
        background_rms=rms,
    )
    parametric_source_arc_masks = [None, None, None] if not USE_SOURCE_MASKS_PARAMETRIC else [None, source1_mask, source2_mask]
    pixelated_source_arc_masks = [None, None, None] if not USE_SOURCE_MASKS_PIXELATED else [None, source1_mask, source2_mask]
    source_scales = [1.0, SOURCE_GRID_SCALE, SOURCE_GRID_SCALE]
    conjugate_points_model = None
    conjugate_points_fits = load_points_from_payload(CONJUGATE_PAYLOAD, 'conjugatePointsFitsCoordinates')
    if conjugate_points_fits is not None:
        if key != REFERENCE_BAND:
            conjugate_points_fits = map_fits_points_between_wcs(conjugate_points_fits, REFERENCE_SCIENCE_WCS, science_wcs)
        conjugate_points_model = pixel_fits_to_model_xy_array(conjugate_points_fits, science_data.shape, pixel_scale)


    parametric_light_model = MPLightModel([
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
    ])
    pixel_kwargs = {
        'pixel_adaptive_grid': True,
        'pixel_interpol': 'fast_bilinear',
        'kwargs_pixelated': {'num_pixels': PIXEL_GRID_SHAPE},
    }
    pixelated_light_model = MPLightModel([
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['PIXELATED'], **pixel_kwargs),
        LightModel(['PIXELATED'], **pixel_kwargs),
    ])
    mass_model = MPMassModel([
        MassModel(['EPL', 'SHEAR']),
        MassModel(['SIS']),
    ])
    lens_image_parametric = MPLensImage(
        pixel_grid,
        psf,
        noise_class=noise,
        light_model_class=parametric_light_model,
        mass_model_class=mass_model,
        source_arc_masks=parametric_source_arc_masks,
        conjugate_points=[jnp.asarray(conjugate_points_model, dtype=jnp.float64), jnp.asarray(conjugate_points_model, dtype=jnp.float64)] if conjugate_points_model is not None else None,
        source_grid_scale=source_scales,
        kwargs_numerics={'supersampling_factor': 1},
    )
    lens_image_pixelated = MPLensImage(
        pixel_grid,
        psf,
        noise_class=noise,
        light_model_class=pixelated_light_model,
        mass_model_class=mass_model,
        source_arc_masks=pixelated_source_arc_masks,
        conjugate_points=[jnp.asarray(conjugate_points_model, dtype=jnp.float64), jnp.asarray(conjugate_points_model, dtype=jnp.float64)] if conjugate_points_model is not None else None,
        source_grid_scale=source_scales,
        kwargs_numerics={'supersampling_factor': 1},
    )
    return BandData(
        key=key,
        filter_name=filter_name,
        data=jnp.asarray(science_data, dtype=jnp.float64),
        obs_mask=jnp.asarray(obs_mask, dtype=bool),
        num_obs=int(obs_mask.sum()),
        source1_mask=source1_mask,
        source2_mask=source2_mask,
        rms=rms,
        pixel_scale=pixel_scale,
        exposure_time=exposure_time,
        science_cutout_path=science_cutout_path,
        source1_mask_cutout_path=INNER_MASK_PATH,
        source2_mask_cutout_path=OUTER_MASK_PATH,
        psf_cutout_path=psf_cutout_path,
        background_box_path=background_box_path,
        science_extent=science_extent,
        psf_extent=psf_extent,
        science_wcs=science_wcs,
        lens_image_parametric=lens_image_parametric,
        lens_image_pixelated=lens_image_pixelated,
        conjugate_points_model=conjugate_points_model,
    )


In [ ]:
K_GRID_PIXEL = PowerSpectrum.K_grid((PIXEL_GRID_SHAPE, PIXEL_GRID_SHAPE))


def scalar_param(params, key):
    return jnp.ravel(jnp.asarray(params[key], dtype=jnp.float64))[0]


def vector_param(params, key):
    return jnp.asarray(params[key], dtype=jnp.float64)


def qphi_to_e1e2(q, phi):
    ellipticity = (1.0 - q) / (1.0 + q)
    return ellipticity * jnp.cos(2.0 * phi), ellipticity * jnp.sin(2.0 * phi)


def get_band(band_key):
    for band in bands:
        if band.key == band_key:
            return band
    raise KeyError(f'Unknown band key: {band_key}')


def get_conjugate_centroid_fits(payload):
    points = payload.get('conjugatePointsFitsCoordinates', [])
    if len(points) == 0:
        raise ValueError(f'No conjugate points found in {CONJUGATE_JSON_PATH}')
    xy = np.array([[point['x'], point['y']] for point in points], dtype=float)
    return jnp.asarray(xy.mean(axis=0), dtype=jnp.float64)


def points_to_model_xy(points_xy, band):
    points_xy = np.asarray(points_xy, dtype=np.float64)
    center_xy = np.array([(band.data.shape[1] - 1) / 2.0, (band.data.shape[0] - 1) / 2.0], dtype=np.float64)
    return (points_xy - center_xy) * band.pixel_scale


def points_to_pixel_xy(points_model, band):
    points_model = np.asarray(points_model, dtype=np.float64)
    center_xy = np.array([(band.data.shape[1] - 1) / 2.0, (band.data.shape[0] - 1) / 2.0], dtype=np.float64)
    return points_model / band.pixel_scale + center_xy


def ensure_points_2d(points):
    points = np.asarray(points, dtype=np.float64)
    if points.ndim == 1:
        return points[np.newaxis, :], True
    return points, False


def map_model_points_between_bands(points_model, source_band_key, target_band_key):
    source_band = get_band(source_band_key)
    target_band = get_band(target_band_key)
    points_model, squeeze_output = ensure_points_2d(points_model)
    source_pixels = points_to_pixel_xy(points_model, source_band)
    world = source_band.science_wcs.pixel_to_world_values(source_pixels[:, 0], source_pixels[:, 1])
    target_pixels = np.column_stack(target_band.science_wcs.world_to_pixel_values(*world))
    target_model = points_to_model_xy(target_pixels, target_band)
    result = jnp.asarray(target_model, dtype=jnp.float64)
    return result[0] if squeeze_output else result


def fit_band_wcs_affine_transform(source_band_key, target_band_key, sample_scale=1.0):
    if source_band_key == target_band_key:
        return {'matrix': np.eye(2, dtype=np.float64), 'offset': np.zeros(2, dtype=np.float64)}
    sample_points = np.array([
        [0.0, 0.0],
        [sample_scale, 0.0],
        [0.0, sample_scale],
    ], dtype=np.float64)
    mapped_points = np.asarray(map_model_points_between_bands(sample_points, source_band_key, target_band_key), dtype=np.float64)
    design = np.column_stack([sample_points, np.ones(sample_points.shape[0], dtype=np.float64)])
    solution, *_ = np.linalg.lstsq(design, mapped_points, rcond=None)
    return {
        'matrix': solution[:2, :].T,
        'offset': solution[2, :],
    }


def apply_wcs_affine_transform(points_model, source_band_key, target_band_key):
    if source_band_key == target_band_key:
        return jnp.asarray(points_model, dtype=jnp.float64)
    transform = BAND_WCS_TRANSFORMS[(source_band_key, target_band_key)]
    matrix = jnp.asarray(transform['matrix'], dtype=jnp.float64)
    offset = jnp.asarray(transform['offset'], dtype=jnp.float64)
    points_model = jnp.asarray(points_model, dtype=jnp.float64)
    if points_model.ndim == 1:
        return matrix @ points_model + offset
    return (points_model @ matrix.T) + offset


def transform_angle_between_bands(center_model, phi, source_band_key, target_band_key, step_arcsec=0.1):
    if source_band_key == target_band_key:
        return jnp.asarray(phi, dtype=jnp.float64)
    direction = jnp.array([jnp.cos(phi), jnp.sin(phi)], dtype=jnp.float64)
    matrix = jnp.asarray(BAND_WCS_TRANSFORMS[(source_band_key, target_band_key)]['matrix'], dtype=jnp.float64)
    mapped_direction = matrix @ direction
    return jnp.arctan2(mapped_direction[1], mapped_direction[0])


def get_band_conjugate_points_model(band_key):
    if CONJUGATE_PAYLOAD is None or CONJUGATE_REFERENCE_BAND is None:
        return None
    reference_band = get_band(CONJUGATE_REFERENCE_BAND)
    points_fits = load_points_from_payload(CONJUGATE_PAYLOAD, 'conjugatePointsFitsCoordinates')
    if points_fits is None:
        return None
    if band_key == CONJUGATE_REFERENCE_BAND:
        return jnp.asarray(points_to_model_xy(points_fits, reference_band), dtype=jnp.float64)
    target_band = get_band(band_key)
    mapped_points = map_fits_points_between_wcs(points_fits, reference_band.science_wcs, target_band.science_wcs)
    return jnp.asarray(points_to_model_xy(mapped_points, target_band), dtype=jnp.float64)


def get_band_conjugate_centroid_model(band_key):
    points = get_band_conjugate_points_model(band_key)
    if points is None or int(points.shape[0]) == 0:
        return None
    return jnp.mean(points, axis=0)


def get_reference_conjugate_point_model(reference_band_key=None):
    if reference_band_key is None:
        reference_band_key = CONJUGATE_REFERENCE_BAND
    centroid = get_band_conjugate_centroid_model(reference_band_key)
    if centroid is None:
        raise ValueError(f'No conjugate points found in {CONJUGATE_JSON_PATH}')
    return centroid


def get_band_trace_points_model(band_key):
    if TRACE_POINT_PAYLOAD is None or TRACE_POINT_REFERENCE_BAND is None:
        return None
    points_fits = load_points_from_payload(TRACE_POINT_PAYLOAD, 'tracePointsFitsCoordinates')
    if points_fits is None:
        return None
    reference_band = get_band(TRACE_POINT_REFERENCE_BAND)
    if band_key == TRACE_POINT_REFERENCE_BAND:
        return jnp.asarray(points_to_model_xy(points_fits, reference_band), dtype=jnp.float64)
    target_band = get_band(band_key)
    mapped_points = map_fits_points_between_wcs(points_fits, reference_band.science_wcs, target_band.science_wcs)
    return jnp.asarray(points_to_model_xy(mapped_points, target_band), dtype=jnp.float64)


def get_wcs_relative_geometry(source_band_key, target_band_key):
    center_source = jnp.array([0.0, 0.0], dtype=jnp.float64)
    center_target = map_model_points_between_bands(center_source, source_band_key, target_band_key)
    phi_target = transform_angle_between_bands(center_source, 0.0, source_band_key, target_band_key)
    return {
        'delta_x_arcsec': float(np.asarray(center_target[0])),
        'delta_y_arcsec': float(np.asarray(center_target[1])),
        'delta_phi_rad': float(np.asarray(phi_target)),
        'delta_phi_deg': float(np.asarray(phi_target) * 180.0 / np.pi),
    }


def sample_shared_main_mass_shape():
    return {
        'theta_E_main': numpyro.sample('theta_E_main', dist.Uniform(PRIORS['theta_E_main']['low'], PRIORS['theta_E_main']['high'])),
        'gamma_main': numpyro.sample('gamma_main', dist.Uniform(PRIORS['gamma_main']['low'], PRIORS['gamma_main']['high'])),
        'q_main': numpyro.sample('q_main', dist.Uniform(PRIORS['q_main']['low'], PRIORS['q_main']['high'])),
        'gamma_shear_main': numpyro.sample('gamma_shear_main', dist.Uniform(PRIORS['gamma_shear_main']['low'], PRIORS['gamma_shear_main']['high']).expand([2])),
    }


def sample_parametric_main_mass_params():
    params = sample_shared_main_mass_shape()
    params['center_main_shared'] = numpyro.sample(
        'center_main_shared',
        dist.TruncatedNormal(0.0, 0.08, low=PRIORS['center_main']['low'], high=PRIORS['center_main']['high']).expand([2]),
    )
    params['phi_main_shared'] = numpyro.sample('phi_main_shared', dist.Uniform(PRIORS['phi_main']['low'], PRIORS['phi_main']['high']))
    return params


def sample_pixelated_main_mass_params():
    params = sample_shared_main_mass_shape()
    params['center_main_shared'] = numpyro.sample(
        'center_main_shared',
        dist.TruncatedNormal(0.0, 0.08, low=PRIORS['center_main']['low'], high=PRIORS['center_main']['high']).expand([2]),
    )
    params['phi_main_shared'] = numpyro.sample('phi_main_shared', dist.Uniform(PRIORS['phi_main']['low'], PRIORS['phi_main']['high']))
    return params


def get_main_center_phi(params, band_key, model_mode):
    center_reference = vector_param(params, 'center_main_shared')
    phi_reference = scalar_param(params, 'phi_main_shared')
    if band_key == REFERENCE_BAND:
        return center_reference, phi_reference
    center_target = apply_wcs_affine_transform(center_reference, REFERENCE_BAND, band_key)
    phi_target = transform_angle_between_bands(center_reference, phi_reference, REFERENCE_BAND, band_key)
    return center_target, phi_target


def build_main_mass_kwargs(params, band_key, model_mode):
    center_xy, phi = get_main_center_phi(params, band_key, model_mode)
    e1, e2 = qphi_to_e1e2(scalar_param(params, 'q_main'), phi)
    shear = vector_param(params, 'gamma_shear_main')
    return [{
        'theta_E': scalar_param(params, 'theta_E_main'),
        'gamma': scalar_param(params, 'gamma_main'),
        'e1': e1,
        'e2': e2,
        'center_x': center_xy[0],
        'center_y': center_xy[1],
    }, {
        'gamma1': shear[0],
        'gamma2': shear[1],
        'ra_0': center_xy[0],
        'dec_0': center_xy[1],
    }]


def trace_conjugate_centroid_to_source1(band, params, kwargs_main_mass, model_mode):
    if band.conjugate_points_model is None or len(band.conjugate_points_model) == 0:
        center_xy, _ = get_main_center_phi(params, band.key, model_mode)
        return center_xy, None
    lens_image = band.lens_image_parametric if model_mode == 'parametric' else band.lens_image_pixelated
    conj_points_at_s1 = lens_image.trace_conjugate_points(
        eta_flat=build_eta_flat(params),
        kwargs_mass=[kwargs_main_mass],
        N=1,
    )
    return conj_points_at_s1.mean(axis=0), conj_points_at_s1


def get_reference_source1_sis_center(params, model_mode):
    reference_band = get_band(REFERENCE_BAND)
    kwargs_main_mass_reference = build_main_mass_kwargs(params, REFERENCE_BAND, model_mode)
    center_s1_reference, conj_points_at_s1_reference = trace_conjugate_centroid_to_source1(
        reference_band,
        params,
        kwargs_main_mass_reference,
        model_mode,
    )
    return center_s1_reference, conj_points_at_s1_reference


def build_source1_sis_kwargs(params, band, model_mode, kwargs_main_mass=None):
    center_s1_reference, _ = get_reference_source1_sis_center(params, model_mode)
    if band.key == REFERENCE_BAND:
        center_s1 = center_s1_reference
    else:
        center_s1 = apply_wcs_affine_transform(center_s1_reference, REFERENCE_BAND, band.key)
    return [{
        'theta_E': scalar_param(params, 'theta_E_source1'),
        'center_x': center_s1[0],
        'center_y': center_s1[1],
    }]


def build_mass_kwargs(params, band, model_mode):
    kwargs_main_mass = build_main_mass_kwargs(params, band.key, model_mode)
    return [
        kwargs_main_mass,
        build_source1_sis_kwargs(params, band, model_mode, kwargs_main_mass),
    ]


def build_eta_flat(params):
    return jnp.atleast_1d(jnp.asarray(params['eta'], dtype=jnp.float64))


def get_trace_point_ids(payload):
    if payload is None:
        return []
    points = payload.get('tracePointsFitsCoordinates', [])
    return [int(point['id']) for point in points]


def get_band_trace_points_subset_model(band_key, point_ids):
    points = get_band_trace_points_model(band_key)
    if points is None:
        return None
    available_ids = get_trace_point_ids(TRACE_POINT_PAYLOAD)
    index_by_id = {point_id: index for index, point_id in enumerate(available_ids)}
    selected_indices = [index_by_id[point_id] for point_id in point_ids if point_id in index_by_id]
    if len(selected_indices) == 0:
        return None
    selected_indices = jnp.asarray(selected_indices, dtype=jnp.int32)
    return jnp.asarray(points[selected_indices], dtype=jnp.float64)


def trace_selected_points_to_source_plane(band, params, kwargs_mass, point_ids, plane_index):
    image_points = get_band_trace_points_subset_model(band.key, point_ids)
    if image_points is None or int(image_points.shape[0]) == 0:
        return None
    x_img = jnp.asarray(image_points[:, 0], dtype=jnp.float64)
    y_img = jnp.asarray(image_points[:, 1], dtype=jnp.float64)
    x_planes, y_planes = band.lens_image_pixelated.MPMassModel.ray_shooting(
        x_img,
        y_img,
        build_eta_flat(params),
        kwargs_mass,
        N=plane_index,
    )
    return jnp.column_stack([x_planes[plane_index], y_planes[plane_index]])


def sample_trace_point_conjugate_prior(band, params, kwargs_mass, plane_index, point_ids, suffix):
    if not USE_TRACE_POINT_CONJUGATE_PRIOR:
        return
    traced_points = trace_selected_points_to_source_plane(band, params, kwargs_mass, point_ids, plane_index)
    if traced_points is None:
        return
    conj_distance = reduced_distance_matrix(traced_points)
    nc = int(conj_distance.shape[0])
    if nc > 0:
        with numpyro.plate(f'Conjugate points {plane_index} {band.key} - [{nc}]', nc):
            numpyro.sample(f'conjugate_points_{suffix}_{band.key}', dist.Exponential(CONJUGATE_PRIOR_RATE), obs=conj_distance)


def reduced_distance_matrix(points):
    points = jnp.asarray(points, dtype=jnp.float64)
    n_points = int(points.shape[0])
    if n_points < 2:
        return jnp.zeros((0,), dtype=jnp.float64)
    deltas = points[:, None, :] - points[None, :, :]
    distances = jnp.sqrt(jnp.sum(deltas**2, axis=-1))
    iu = jnp.triu_indices(n_points, k=1)
    return distances[iu]


def pixelize_parametric_source(band, params, plane_index):
    kwargs_mass = build_mass_kwargs(params, band, 'parametric')
    kwargs_light = Light.band_from_parametric_params(params, band.key)
    eta_flat = build_eta_flat(params)
    x_axes, y_axes, _ = band.lens_image_pixelated.get_source_coordinates(
        eta_flat,
        kwargs_mass,
        force=True,
        npix_src=PIXEL_GRID_SHAPE,
        source_grid_scale=SOURCE_GRID_SCALE,
    )
    x_grid, y_grid = jnp.meshgrid(x_axes[plane_index], y_axes[plane_index])
    source_image = band.lens_image_parametric.MPLightModel.light_models[plane_index].surface_brightness(
        x_grid,
        y_grid,
        kwargs_light[plane_index],
        pixels_x_coord=x_axes[plane_index],
        pixels_y_coord=y_axes[plane_index],
    )
    return source_image * band.lens_image_parametric.Grid.pixel_area


def fit_power_spectrum_init(image, rng_key, param_name, max_iterations=STAGE2_INIT_STEPS):
    image = jnp.asarray(image, dtype=jnp.float64)
    noise = jnp.maximum(1e-6, 0.001 * jnp.max(image))

    def power_spectrum_init_model(image_in, noise_in, k_values):
        ps_model = PowerSpectrum.matern_power_spectrum(
            f'Init {param_name}',
            param_name,
            k_values,
            n_value=None,
            positive=True,
        )
        ny, nx = image_in.shape
        with numpyro.plate(f'{param_name} data x - [{nx}]', nx):
            with numpyro.plate(f'{param_name} data y - [{ny}]', ny):
                numpyro.sample(
                    f'obs_{param_name}',
                    dist.Normal(ps_model['pixels'], noise_in),
                    obs=image_in,
                )

    guide = autoguide.AutoDiagonalNormal(power_spectrum_init_model, init_loc_fn=infer.init_to_median())
    scheduler = split_scheduler(
        max_iterations,
        init_value=0.01,
        transition_steps=[50, 10],
    )
    svi = infer.SVI(
        power_spectrum_init_model,
        guide,
        optax.adabelief(learning_rate=scheduler),
        infer.TraceMeanField_ELBO(),
    )
    result = svi.run(
        rng_key,
        max_iterations,
        image,
        noise,
        K_GRID_PIXEL.k,
        progress_bar=False,
        stable_update=True,
    )
    return guide.median(result.params)


def build_stage2_init_values(parametric_params, rng_key):
    stage2_init_values = {
        'eta': parametric_params['eta'],
        'theta_E_main': parametric_params['theta_E_main'],
        'gamma_main': parametric_params['gamma_main'],
        'q_main': parametric_params['q_main'],
        'gamma_shear_main': parametric_params['gamma_shear_main'],
        'theta_E_source1': parametric_params['theta_E_source1'],
    }
    for band in bands:
        stage2_init_values[f'A_lens_{band.key}'] = parametric_params[f'A_lens_{band.key}']
        stage2_init_values[f'sigma_lens_{band.key}'] = parametric_params[f'sigma_lens_{band.key}']
        stage2_init_values[f'e_lens_{band.key}'] = parametric_params[f'e_lens_{band.key}']
        stage2_init_values[f'center_lens_{band.key}'] = parametric_params[f'center_lens_{band.key}']
    stage2_init_values['center_main_shared'] = parametric_params['center_main_shared']
    stage2_init_values['phi_main_shared'] = parametric_params['phi_main_shared']
    for band in bands:
        stage2_init_values[f'RMS_{band.key}'] = parametric_params[f'RMS_{band.key}']
    fit_key = rng_key
    for band in bands:
        for plane_index, param_name in [
            (1, f'source1pix_{band.key}'),
            (2, f'source2pix_{band.key}'),
        ]:
            fit_key, ps_key = jax.random.split(fit_key)
            source_image = pixelize_parametric_source(band, parametric_params, plane_index)
            ps_init = fit_power_spectrum_init(source_image, ps_key, param_name)
            stage2_init_values.update(ps_init)
    return stage2_init_values


def build_parametric_model():
    sigma_lims_lens = (0.01, 1.0)
    sigma_lims_source = (0.01, 0.5)

    def model():
        model_params = sample_parametric_main_mass_params()
        model_params['eta'] = numpyro.sample('eta', dist.Uniform(PRIORS['eta']['low'], PRIORS['eta']['high']))
        model_params['theta_E_source1'] = numpyro.sample('theta_E_source1', dist.Uniform(PRIORS['theta_E_source1']['low'], PRIORS['theta_E_source1']['high']))
        eta_flat = build_eta_flat(model_params)

        for band in bands:
            kwargs_main_mass = build_main_mass_kwargs(model_params, band.key, 'parametric')
            center_s1_reference, conj_points_at_s1_reference = get_reference_source1_sis_center(model_params, 'parametric')
            if band.key == REFERENCE_BAND:
                center_s1 = center_s1_reference
                conj_points_at_s1 = conj_points_at_s1_reference
            else:
                center_s1 = apply_wcs_affine_transform(center_s1_reference, REFERENCE_BAND, band.key)
                conj_points_at_s1 = None
            kwargs_source1_mass = [{
                'theta_E': scalar_param(model_params, 'theta_E_source1'),
                'center_x': center_s1[0],
                'center_y': center_s1[1],
            }]
            kwargs_mass = [kwargs_main_mass, kwargs_source1_mass]
            sample_trace_point_conjugate_prior(
                band,
                model_params,
                kwargs_mass,
                plane_index=1,
                point_ids=TRACE_CONJUGATE_GROUPS['source1'],
                suffix='source1',
            )
            sample_trace_point_conjugate_prior(
                band,
                model_params,
                kwargs_mass,
                plane_index=2,
                point_ids=TRACE_CONJUGATE_GROUPS['source2'],
                suffix='source2',
            )

            lens_light = Light.multi_gauss_light(
                f'{band.key} lens light',
                f'lens_{band.key}',
                N_GAUSS_LENS,
                sigma_lims_lens,
                center_low=-0.2,
                center_high=0.2,
                e_low=-0.2,
                e_high=0.2,
            )
            source_1_light = Light.multi_gauss_light(
                f'{band.key} source 1 light',
                f'source1_{band.key}',
                N_GAUSS_SOURCE,
                sigma_lims_source,
            )
            source_2_light = Light.multi_gauss_light(
                f'{band.key} source 2 light',
                f'source2_{band.key}',
                N_GAUSS_SOURCE,
                sigma_lims_source,
            )
            kwargs_light = [lens_light, source_1_light, source_2_light]
            model_image = band.lens_image_parametric.model(
                eta_flat=eta_flat,
                kwargs_mass=kwargs_mass,
                kwargs_light=kwargs_light,
            )
            rms_low = max(0.8 * band.rms, 1e-6)
            rms_high = max(1.2 * band.rms, rms_low * 1.05)
            background_rms = numpyro.sample(f'RMS_{band.key}', dist.LogUniform(rms_low, rms_high))
            model_var = band.lens_image_parametric.Noise.C_D_model(model_image, background_rms=background_rms)
            model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))
            numpyro.deterministic(f'model_{band.key}', model_image)
            with numpyro.plate(f'obs_{band.key} - [{band.num_obs}]', band.num_obs):
                numpyro.sample(
                    f'obs_{band.key}',
                    dist.Normal(model_image[band.obs_mask], model_std[band.obs_mask]),
                    obs=band.data[band.obs_mask],
                )

    return model


def build_pixelated_model(fixed_lens_lights=None):
    sigma_lims_lens = (0.01, 1.0)

    def model():
        model_params = sample_pixelated_main_mass_params()
        model_params['eta'] = numpyro.sample('eta', dist.Uniform(PRIORS['eta']['low'], PRIORS['eta']['high']))
        model_params['theta_E_source1'] = numpyro.sample('theta_E_source1', dist.Uniform(PRIORS['theta_E_source1']['low'], PRIORS['theta_E_source1']['high']))
        eta_flat = build_eta_flat(model_params)

        for band in bands:
            kwargs_main_mass = build_main_mass_kwargs(model_params, band.key, 'pixelated')
            kwargs_source1_mass = build_source1_sis_kwargs(model_params, band, 'pixelated', kwargs_main_mass)
            kwargs_mass = [kwargs_main_mass, kwargs_source1_mass]
            lens_light = Light.multi_gauss_light(
                f'{band.key} lens light',
                f'lens_{band.key}',
                N_GAUSS_LENS,
                sigma_lims_lens,
                center_low=-0.2,
                center_high=0.2,
                e_low=-0.2,
                e_high=0.2,
            )
            source_1_light = PowerSpectrum.matern_power_spectrum(
                f'{band.key} source 1 power',
                f'source1pix_{band.key}',
                K_GRID_PIXEL.k,
                k_zero=0.0,
            )
            source_2_light = PowerSpectrum.matern_power_spectrum(
                f'{band.key} source 2 power',
                f'source2pix_{band.key}',
                K_GRID_PIXEL.k,
                k_zero=0.0,
            )
            kwargs_light = [
                lens_light,
                [source_1_light],
                [source_2_light],
            ]
            sample_trace_point_conjugate_prior(
                band,
                model_params,
                kwargs_mass,
                plane_index=1,
                point_ids=TRACE_CONJUGATE_GROUPS['source1'],
                suffix='source1',
            )
            sample_trace_point_conjugate_prior(
                band,
                model_params,
                kwargs_mass,
                plane_index=2,
                point_ids=TRACE_CONJUGATE_GROUPS['source2'],
                suffix='source2',
            )
            model_image = band.lens_image_pixelated.model(
                eta_flat=eta_flat,
                kwargs_mass=kwargs_mass,
                kwargs_light=kwargs_light,
            )
            rms_low = max(0.5 * band.rms, 1e-6)
            rms_high = max(2.0 * band.rms, rms_low * 1.5)
            background_rms = numpyro.sample(f'RMS_{band.key}', dist.LogUniform(rms_low, rms_high))
            model_var = band.lens_image_pixelated.Noise.C_D_model(model_image, background_rms=background_rms)
            model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))
            numpyro.deterministic(f'model_{band.key}', model_image)
            with numpyro.plate(f'obs_{band.key} - [{band.num_obs}]', band.num_obs):
                numpyro.sample(
                    f'obs_{band.key}',
                    dist.Normal(model_image[band.obs_mask], model_std[band.obs_mask]),
                    obs=band.data[band.obs_mask],
                )

    return model


def build_model(model_mode, bands, fixed_lens_lights=None):
    if model_mode == 'parametric':
        return build_parametric_model()
    if model_mode == 'pixelated':
        return build_pixelated_model(fixed_lens_lights)
    raise ValueError(f'Unknown model_mode: {model_mode}')


def run_svi(model, rng_key, steps, init_values=None):
    init_fn = infer.init_to_median(num_samples=25)
    if init_values:
        init_fn = ResumeInit.init_to_value_or_defer(values=init_values, defer=init_fn)
    guide = autoguide.AutoLowRankMultivariateNormal(model, init_loc_fn=init_fn)
    scheduler = split_scheduler(
        max_iterations=max(steps, 2),
        init_value=0.01,
        transition_steps=[200, 10],
    )
    optim = optax.adabelief(learning_rate=scheduler)
    svi = infer.SVI(model, guide, optim, infer.Trace_ELBO(num_particles=10))
    result = svi.run(rng_key, steps, progress_bar=True, stable_update=True)
    return result, guide.median(result.params)


def build_light_kwargs_from_params(params, band_key, model_mode, fixed_lens_lights=None):
    if model_mode == 'parametric':
        return Light.band_from_parametric_params(params, band_key)
    return [
        Light.params2kwargs_multi_gauss_light(params, f'lens_{band_key}'),
        [PowerSpectrum.params2kwargs(params, f'source1pix_{band_key}')],
        [PowerSpectrum.params2kwargs(params, f'source2pix_{band_key}')],
    ]


def positive_log_norm(image, lower_percentile=5.0, upper_percentile=99.5, floor=1e-6):
    image = np.asarray(image, dtype=float)
    positive = image[np.isfinite(image) & (image > 0)]
    if positive.size == 0:
        return None
    vmin = max(float(np.nanpercentile(positive, lower_percentile)), floor)
    vmax = max(float(np.nanpercentile(positive, upper_percentile)), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)


def symmetric_display_limits(image, percentile=99.0, floor=1e-6):
    image = np.asarray(image, dtype=float)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        vmax = 1.0
    else:
        vmax = max(float(np.nanpercentile(np.abs(finite), percentile)), floor)
    return {'vmin': -vmax, 'vmax': vmax}


def imshow_panel(ax, image, extent, title, cmap='twilight', norm=None, limits=None):
    image = np.asarray(image)
    if limits is not None:
        ax.imshow(image, origin='lower', cmap=cmap, norm=norm, extent=extent, vmin=limits['vmin'], vmax=limits['vmax'])
    else:
        ax.imshow(image, origin='lower', cmap=cmap, norm=norm, extent=extent)
    ax.set_xlabel('x [arcsec]')
    ax.set_ylabel('y [arcsec]')
    ax.set_title(title)


def get_pixelated_source_image_and_extent(band, params, kwargs_mass, plane_index):
    eta_flat = build_eta_flat(params)
    x_axes, y_axes, _ = band.lens_image_pixelated.get_source_coordinates(
        eta_flat,
        kwargs_mass,
        force=True,
        npix_src=PIXEL_GRID_SHAPE,
        source_grid_scale=SOURCE_GRID_SCALE,
    )
    source_pixels = np.asarray(params[f'pixels_source{plane_index}pix_{band.key}'])
    extent = (
        float(np.min(x_axes[plane_index])),
        float(np.max(x_axes[plane_index])),
        float(np.min(y_axes[plane_index])),
        float(np.max(y_axes[plane_index])),
    )
    return source_pixels, extent


def get_pixelated_image_plane_components(band, params, kwargs_mass, kwargs_light):
    lens_image = band.lens_image_pixelated
    model_kwargs = {
        'eta_flat': build_eta_flat(params),
        'kwargs_mass': kwargs_mass,
        'kwargs_light': kwargs_light,
    }
    lens_light_image = np.asarray(lens_image.model(**model_kwargs, k_planes=(0,), apply_mask=False))
    source_1_lensed_arc = np.asarray(lens_image.model(**model_kwargs, k_planes=(1,)))
    source_2_lensed_arc = np.asarray(lens_image.model(**model_kwargs, k_planes=(2,)))
    return lens_light_image, source_1_lensed_arc, source_2_lensed_arc


def mask_pixel_coordinates(mask, extent):
    rows, cols = np.nonzero(mask)
    if rows.size == 0:
        return np.empty((0,), dtype=float), np.empty((0,), dtype=float)
    ny, nx = mask.shape
    x_axis = np.linspace(extent[0], extent[1], nx)
    y_axis = np.linspace(extent[2], extent[3], ny)
    return x_axis[cols], y_axis[rows]


def trace_mask_to_source_plane(band, params, kwargs_mass, mask, plane_index):
    x_img, y_img = mask_pixel_coordinates(mask, band.science_extent)
    if x_img.size == 0:
        return np.empty((0,), dtype=float), np.empty((0,), dtype=float)
    x_planes, y_planes = band.lens_image_pixelated.MPMassModel.ray_shooting(
        jnp.asarray(x_img, dtype=jnp.float64),
        jnp.asarray(y_img, dtype=jnp.float64),
        build_eta_flat(params),
        kwargs_mass,
        N=plane_index,
    )
    return np.asarray(x_planes[plane_index]), np.asarray(y_planes[plane_index])


def trace_points_to_source_planes(band, params, kwargs_mass, model_mode):
    image_points = get_band_trace_points_model(band.key)
    if image_points is None or int(image_points.shape[0]) == 0:
        return None, None, None
    x_img = jnp.asarray(image_points[:, 0], dtype=jnp.float64)
    y_img = jnp.asarray(image_points[:, 1], dtype=jnp.float64)
    x_planes, y_planes = band.lens_image_pixelated.MPMassModel.ray_shooting(
        x_img,
        y_img,
        build_eta_flat(params),
        kwargs_mass,
        N=2,
    )
    source_1_points = np.column_stack([np.asarray(x_planes[1]), np.asarray(y_planes[1])])
    source_2_points = np.column_stack([np.asarray(x_planes[2]), np.asarray(y_planes[2])])
    return np.asarray(image_points), source_1_points, source_2_points


def annotate_numbered_points(ax, points, color, text_offset=0.02):
    if points is None or len(points) == 0:
        return
    points = np.asarray(points, dtype=float)
    ax.scatter(points[:, 0], points[:, 1], s=18, c=color, marker='x', linewidths=1.0, zorder=5)
    for index, (x_coord, y_coord) in enumerate(points, start=1):
        ax.text(x_coord + text_offset, y_coord + text_offset, str(index), color=color, fontsize=8, weight='bold', zorder=6)


def plot_stage_results(title, model_mode, params, bands, fixed_lens_lights=None):
    if model_mode == 'pixelated':
        fig, axes = plt.subplots(3 * len(bands), 3, figsize=(12, 11 * len(bands)), squeeze=False)
    else:
        fig, axes = plt.subplots(len(bands), 3, figsize=(12, 4 * len(bands)), squeeze=False)
    for band_index, band in enumerate(bands):
        science_x_axis, science_y_axis = extent_axes(band.science_extent, np.asarray(band.data).shape)
        lens_image = band.lens_image_parametric if model_mode == 'parametric' else band.lens_image_pixelated
        kwargs_mass = build_mass_kwargs(params, band, model_mode)
        kwargs_light = build_light_kwargs_from_params(params, band.key, model_mode, fixed_lens_lights=fixed_lens_lights)
        model_image = np.asarray(lens_image.model(
            eta_flat=build_eta_flat(params),
            kwargs_mass=kwargs_mass,
            kwargs_light=kwargs_light,
        ))
        background_rms = float(np.asarray(scalar_param(params, f'RMS_{band.key}'))) if f'RMS_{band.key}' in params else float(band.rms)
        model_var = np.asarray(lens_image.Noise.C_D_model(model_image, background_rms=background_rms))
        model_std = np.sqrt(np.maximum(model_var, 1e-12))
        residual = (np.asarray(band.data) - model_image) / model_std

        if model_mode == 'parametric':
            row_axes = axes[band_index]
            panel_specs = [
                ('data', np.asarray(band.data), band.science_extent, 'twilight', positive_log_norm(band.data), None),
                ('model', model_image, band.science_extent, 'twilight', positive_log_norm(model_image), None),
                ('residual / noise', residual, band.science_extent, 'bwr', None, {'vmin': -3, 'vmax': 3}),
            ]
            for col, (panel_title, image, panel_extent, cmap, norm, limits) in enumerate(panel_specs):
                imshow_panel(row_axes[col], image, panel_extent, f'{band.filter_name} {panel_title}', cmap=cmap, norm=norm, limits=limits)
                if panel_title == 'data':
                    row_axes[col].contour(science_x_axis, science_y_axis, band.source1_mask.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--')
                    row_axes[col].contour(science_x_axis, science_y_axis, band.source2_mask.astype(float), levels=[0.5], colors='orange', linewidths=1.0, linestyles='--')
            continue

        lens_light_image, source_1_lensed_arc, source_2_lensed_arc = get_pixelated_image_plane_components(
            band,
            params,
            kwargs_mass,
            kwargs_light,
        )
        lens_light_subtracted_data = np.asarray(band.data) - lens_light_image
        source_1_image, extent_s1 = get_pixelated_source_image_and_extent(band, params, kwargs_mass, 1)
        source_2_image, extent_s2 = get_pixelated_source_image_and_extent(band, params, kwargs_mass, 2)
        source_1_trace = trace_mask_to_source_plane(band, params, kwargs_mass, band.source1_mask, 1)
        source_2_trace = trace_mask_to_source_plane(band, params, kwargs_mass, band.source2_mask, 2)
        trace_points_lens, trace_points_source_1, trace_points_source_2 = trace_points_to_source_planes(
            band,
            params,
            kwargs_mass,
            model_mode,
        )
        row_specs = [
            [
                ('data', np.asarray(band.data), band.science_extent, 'twilight', positive_log_norm(band.data), None),
                ('model', model_image, band.science_extent, 'twilight', positive_log_norm(model_image), None),
                ('residual / noise', residual, band.science_extent, 'bwr', None, {'vmin': -3, 'vmax': 3}),
            ],
            [
                ('lens light', lens_light_image, band.science_extent, 'twilight', positive_log_norm(lens_light_image), None),
                ('source 1', source_1_image, extent_s1, 'twilight', positive_log_norm(source_1_image), None),
                ('source 2', source_2_image, extent_s2, 'twilight', positive_log_norm(source_2_image), None),
            ],
            [
                ('lens-light subtracted data', lens_light_subtracted_data, band.science_extent, 'bwr', None, symmetric_display_limits(lens_light_subtracted_data)),
                ('source 1 lensed arc', source_1_lensed_arc, band.science_extent, 'twilight', positive_log_norm(source_1_lensed_arc), None),
                ('source 2 lensed arc', source_2_lensed_arc, band.science_extent, 'twilight', positive_log_norm(source_2_lensed_arc), None),
            ],
        ]
        for row_offset, panel_specs in enumerate(row_specs):
            for col, (panel_title, image, panel_extent, cmap, norm, limits) in enumerate(panel_specs):
                ax = axes[3 * band_index + row_offset, col]
                imshow_panel(ax, image, panel_extent, f'{band.filter_name} {panel_title}', cmap=cmap, norm=norm, limits=limits)
                if panel_title in ('data', 'lens-light subtracted data'):
                    ax.contour(science_x_axis, science_y_axis, band.source1_mask.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--')
                    ax.contour(science_x_axis, science_y_axis, band.source2_mask.astype(float), levels=[0.5], colors='orange', linewidths=1.0, linestyles='--')
                if panel_title == 'data':
                    annotate_numbered_points(ax, trace_points_lens, color='yellow')
                if panel_title == 'source 1' and source_1_trace is not None and source_1_trace[0].size > 0:
                    ax.scatter(source_1_trace[0], source_1_trace[1], s=2, c='cyan', alpha=0.25, linewidths=0)
                    annotate_numbered_points(ax, trace_points_source_1, color='yellow')
                if panel_title == 'source 2' and source_2_trace is not None and source_2_trace[0].size > 0:
                    ax.scatter(source_2_trace[0], source_2_trace[1], s=2, c='orange', alpha=0.25, linewidths=0)
                    annotate_numbered_points(ax, trace_points_source_2, color='yellow')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def main_mass_summary(params, model_mode):
    summary = {
        'theta_E': float(np.asarray(scalar_param(params, 'theta_E_main'))),
        'gamma': float(np.asarray(scalar_param(params, 'gamma_main'))),
        'q': float(np.asarray(scalar_param(params, 'q_main'))),
        'gamma1': float(np.asarray(vector_param(params, 'gamma_shear_main')[0])),
        'gamma2': float(np.asarray(vector_param(params, 'gamma_shear_main')[1])),
        'bands': {},
    }
    for band in ACTIVE_BANDS:
        center_xy, phi = get_main_center_phi(params, band, model_mode)
        sis_kwargs = build_source1_sis_kwargs(params, get_band(band), model_mode)[0]
        summary['bands'][band] = {
            'center_x': float(np.asarray(center_xy[0])),
            'center_y': float(np.asarray(center_xy[1])),
            'phi_rad': float(np.asarray(phi)),
            'phi_deg': float(np.asarray(phi) * 180.0 / np.pi),
            'source1_sis_center_x': float(np.asarray(sis_kwargs['center_x'])),
            'source1_sis_center_y': float(np.asarray(sis_kwargs['center_y'])),
        }
    return summary




In [ ]:
bands = [build_band_data(band_key) for band_key in ACTIVE_BANDS]
BAND_WCS_TRANSFORMS = {
    (source_key, target_key): fit_band_wcs_affine_transform(source_key, target_key)
    for source_key in ACTIVE_BANDS
    for target_key in ACTIVE_BANDS
}
conjugate_centroid_fits = np.asarray(get_conjugate_centroid_fits(CONJUGATE_PAYLOAD), dtype=float)
conjugate_centroid_model_by_band = {
    band.key: None if get_band_conjugate_centroid_model(band.key) is None else np.asarray(get_band_conjugate_centroid_model(band.key), dtype=float)
    for band in bands
}
trace_points_model_by_band = {
    band.key: None if get_band_trace_points_model(band.key) is None else np.asarray(get_band_trace_points_model(band.key), dtype=float)
    for band in bands
}
wcs_relative_geometry = {
    band.key: get_wcs_relative_geometry(REFERENCE_BAND, band.key)
    for band in bands if band.key != REFERENCE_BAND
}

fig, axes = plt.subplots(len(bands), 2, figsize=(10, 4 * len(bands)))
if len(bands) == 1:
    axes = np.array([axes])
for row, band in enumerate(bands):
    science_image = np.asarray(band.data, dtype=float)
    science_x_axis, science_y_axis = extent_axes(band.science_extent, science_image.shape)
    finite_science = science_image[np.isfinite(science_image) & (science_image > 0)]
    science_vmin = max(float(np.nanpercentile(finite_science, 5)), 1e-6) if finite_science.size else 1e-6
    science_vmax = max(float(np.nanpercentile(finite_science, 99.5)), science_vmin * 1.01) if finite_science.size else 1.0
    axes[row, 0].imshow(
        science_image,
        origin='lower',
        cmap='twilight',
        extent=band.science_extent,
        norm=LogNorm(vmin=science_vmin, vmax=science_vmax),
    )
    axes[row, 0].contour(science_x_axis, science_y_axis, band.source1_mask.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--')
    axes[row, 0].contour(science_x_axis, science_y_axis, band.source2_mask.astype(float), levels=[0.5], colors='orange', linewidths=1.0, linestyles='--')
    centroid_model = conjugate_centroid_model_by_band[band.key]
    if centroid_model is not None:
        axes[row, 0].scatter(centroid_model[0], centroid_model[1], s=30, c='yellow', marker='x')
    annotate_numbered_points(axes[row, 0], trace_points_model_by_band[band.key], color='yellow')
    axes[row, 0].set_title(f'{band.filter_name} science')
    axes[row, 0].set_xlabel('x [arcsec]')
    axes[row, 0].set_ylabel('y [arcsec]')
    with fits.open(band.psf_cutout_path) as hdul:
        psf_image = np.asarray(hdul[0].data, dtype=float)
        finite_psf = psf_image[np.isfinite(psf_image) & (psf_image > 0)]
        psf_vmin = max(float(np.nanpercentile(finite_psf, 5)), 1e-8) if finite_psf.size else 1e-8
        psf_vmax = max(float(np.nanpercentile(finite_psf, 99.5)), psf_vmin * 1.01) if finite_psf.size else 1.0
        axes[row, 1].imshow(
            psf_image,
            origin='lower',
            cmap='twilight',
            extent=band.psf_extent,
            norm=LogNorm(vmin=psf_vmin, vmax=psf_vmax),
        )
    axes[row, 1].set_title(f'{band.filter_name} PSF')
    axes[row, 1].set_xlabel('x [arcsec]')
    axes[row, 1].set_ylabel('y [arcsec]')
plt.tight_layout()
plt.show()
for band in bands:
    print(band.filter_name)
    print('  science   :', band.science_cutout_path)
    print('  source1   :', band.source1_mask_cutout_path)
    print('  source2   :', band.source2_mask_cutout_path)
    print('  psf       :', band.psf_cutout_path)
    print('  background:', band.background_box_path)
print(f'reference conjugate centroid fits ({CONJUGATE_REFERENCE_BAND} pix):', conjugate_centroid_fits.tolist())
for band in bands:
    print(f'{band.key} conjugate centroid model [arcsec]:', None if conjugate_centroid_model_by_band[band.key] is None else conjugate_centroid_model_by_band[band.key].tolist())
print('trace point json:', TRACE_POINT_JSON_PATH if TRACE_POINT_PAYLOAD is not None else 'not found')
for band in bands:
    if trace_points_model_by_band[band.key] is not None:
        print(f'{band.key} trace points model [arcsec]:', trace_points_model_by_band[band.key].tolist())
print('wcs relative geometry from reference band:')
print(wcs_relative_geometry)



In [ ]:
MODEL_MODE = 'parametric'
rng_key = jax.random.PRNGKey(RNG_SEED)
rng_key, stage1_key, stage2_init_key, stage2_key = jax.random.split(rng_key, 4)
model_parametric = build_model(MODEL_MODE, bands)
parametric_result, parametric_median = run_svi(model_parametric, stage1_key, PARAMETRIC_STEPS)
print('parametric loss =', float(parametric_result.losses[-1]))
print('parametric eta  =', float(np.asarray(parametric_median['eta'])))


In [ ]:
plot_stage_results(
    title='Parametric DSPL Smoke Test',
    model_mode='parametric',
    params=parametric_median,
    bands=bands,
)


In [ ]:
MODEL_MODE = 'pixelated'
model_pixelated = build_model(MODEL_MODE, bands)
stage2_init_values = build_stage2_init_values(parametric_median, stage2_init_key)
pixelated_result, pixelated_median = run_svi(model_pixelated, stage2_key, PIXELATED_STEPS, init_values=stage2_init_values)
print('pixelated loss =', float(pixelated_result.losses[-1]))
print('pixelated eta  =', float(np.asarray(pixelated_median['eta'])))


In [ ]:
plot_stage_results(
    title='Pixelated DSPL Smoke Test',
    model_mode='pixelated',
    params=pixelated_median,
    bands=bands,
)
summary = {
    'cutout_size': CUTOUT_SIZE,
    'psf_cutout_size': list(PSF_CUTOUT_SIZE),
    'pixel_grid_shape': PIXEL_GRID_SHAPE,
    'priors': PRIORS,
    'eta_parametric': float(np.asarray(parametric_median['eta'])),
    'eta_pixelated': float(np.asarray(pixelated_median['eta'])),
    'parametric_loss_final': float(parametric_result.losses[-1]),
    'pixelated_loss_final': float(pixelated_result.losses[-1]),
    'reference_band': REFERENCE_BAND,
    'trace_conjugate_groups': TRACE_CONJUGATE_GROUPS,
    'use_trace_point_conjugate_prior': USE_TRACE_POINT_CONJUGATE_PRIOR,
    'conjugate_prior_rate': CONJUGATE_PRIOR_RATE,
    'conjugate_json_path': str(CONJUGATE_JSON_PATH),
    'trace_point_json_path': str(TRACE_POINT_JSON_PATH) if TRACE_POINT_PAYLOAD is not None else None,
    'conjugate_centroid_fits_xy': list(map(float, np.asarray(get_conjugate_centroid_fits(CONJUGATE_PAYLOAD)))),
    'conjugate_centroid_model_arcsec_by_band': {
        band.key: None if get_band_conjugate_centroid_model(band.key) is None else np.asarray(get_band_conjugate_centroid_model(band.key)).astype(float).tolist()
        for band in bands
    },
    'trace_points_model_arcsec_by_band': {
        band.key: None if get_band_trace_points_model(band.key) is None else np.asarray(get_band_trace_points_model(band.key)).astype(float).tolist()
        for band in bands
    },
    'wcs_relative_geometry_from_reference_band': {
        band.key: get_wcs_relative_geometry(REFERENCE_BAND, band.key)
        for band in bands if band.key != REFERENCE_BAND
    },
    'bands': {
        band.key: {
            'filter': band.filter_name,
            'rms': band.rms,
            'pixel_scale': band.pixel_scale,
            'science_center_pix': list(PREPROCESS_META['cutouts'][band.key]['center_pix']),
            'background_box_center_pix': [
                PREPROCESS_META['cutouts'][band.key]['background_box_info']['center_x'],
                PREPROCESS_META['cutouts'][band.key]['background_box_info']['center_y'],
            ],
            'science_cutout_path': str(band.science_cutout_path),
            'source1_mask_cutout_path': str(band.source1_mask_cutout_path),
            'source2_mask_cutout_path': str(band.source2_mask_cutout_path),
            'psf_cutout_path': str(band.psf_cutout_path),
        }
        for band in bands
    },
    'parametric_main_mass': main_mass_summary(parametric_median, 'parametric'),
    'pixelated_main_mass': main_mass_summary(pixelated_median, 'pixelated'),
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / 'smoke_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(summary)
print('summary:', summary_path)


In [ ]:
band.obs_mask